# 10.2 · Hybrid search

**Owner:** Member 5 · **Branch:** `dev-hybrid` · **Code:** `src/discovery/rank/hybrid.py` → class `HybridRanker` · **Writes to:** `eval/results/hybrid.json`

**What it means:** Ask BM25 and semantic search for their top 100 each, then combine. **Weighted fusion** rescales both score lists to 0–1 and adds them with a weight α. **Reciprocal Rank Fusion (RRF)** ignores scores and adds 1 / (c + rank) from each list.

**Key idea:** BM25 is precise on exact words, semantic search on meaning; a repository both agree on is usually the right one. RRF needs no score rescaling, which is why it is hard to beat. Depends on Members 3 and 4: agree the interface with them early.

Full guide with starter code, and the Results sheet: [team plan](https://claude.ai/artifact/Rst8GHxAbeHavg4JS2RZV3) (tab 10.2 Hybrid search). Handoff and setup: `docs/handoff-methods.md`.

### Settings to try
| Setting | Values | Why |
|---|---|---|
| α (weighted) | 0.2, 0.4, 0.5, 0.6, 0.8 | Weight of BM25; 1 − α goes to semantic. |
| c (RRF) | 10, 60 | Larger c flattens the difference between rank 1 and rank 10. |
| depth | 50, 100 | How many results to take from each method before fusing. |

**Fixed:** inputs are the BM25Ranker and EmbeddingRanker objects through the shared interface; until they are ready, use rank-bm25 and a quick MiniLM over name + description as stand-ins

### What we agreed (same for all five)
- **One tokenizer:** `tokenize()` from `discovery.preprocess.text`, for documents *and* queries. Never a private one.
- **One interface:** your class extends `SearchMethod` (`src/discovery/rank/base.py`): `fit(corpus)` → self, `search(query, k)` → `[{"doc_id": full_name, "score": float}]`, best first. `check_hits()` checks it.
- **Tune on the 5 development queries only** (`DEV`, replaced by the real ones in Phase 13). The 20 test queries are for the final number only.
- **Seed 42**, paths from `PATHS`, and don't edit the shared files (`rank/base.py`, `rank/devkit.py`, `preprocess/text.py`, `config/config.yaml`) on your branch: ask Member 6.
- Commit this notebook **with its outputs** on `dev-hybrid`. No data files in git.

## 1. Setup: load the corpus and the shared tokenizer

**Check:** it prints 8,223 repositories, and `node.js` and `c#` come out of `tokenize()` in one piece.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from discovery.config import PATHS, SEED, set_seed
from discovery.preprocess.text import doc_text, tokenize      # the ONE shared tokenizer (Phase 6)
from discovery.rank.base import check_hits
from discovery.rank.devkit import DEV, load_corpus, measure, save_results, show_top
from discovery.rank.hybrid import HybridRanker

set_seed()
corpus = load_corpus()                                   # data/processed/corpus.parquet
ids = corpus["full_name"].tolist()                       # doc_id = full_name, "owner/name"
text = doc_text(corpus).tolist()                         # name + description + topics + README
print(f"{len(corpus):,} repositories |", tokenize("Built with Node.js and C#"))
print("development queries:", *DEV, sep="\n  ")

## 2. Build: Hybrid search

Hybrid has nothing of its own to build: it reuses the two other methods through `search()`.

**Check:** it runs once on the whole corpus without errors. Write down how long it took: it goes in the sheet.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
from collections import defaultdict

# Members 3 and 4's classes. Until they are in src/, wrap rank_bm25.BM25Okapi and a quick MiniLM
# over name + description in the same search() interface: swapping later is one line.
from discovery.rank.bm25 import BM25Ranker
from discovery.rank.embeddings import EmbeddingRanker
bm25 = BM25Ranker().fit(corpus)
semantic = EmbeddingRanker().fit(corpus)

def minmax(hits):
    if not hits: return {}
    s = [h["score"] for h in hits]; lo, hi = min(s), max(s)
    return {h["doc_id"]: (h["score"] - lo) / (hi - lo) if hi > lo else 1.0 for h in hits}
```
</details>

## 3. Search: a free-text query in, a ranked list out

Two fusion rules. A repository missing from one list gets 0 from it in weighted fusion and nothing in RRF.

**Check:** the top 10 for `DEV[0]` look plausible when you read their names and descriptions.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
def weighted(query, k=10, alpha=0.5, depth=100):
    a, b = minmax(bm25.search(query, depth)), minmax(semantic.search(query, depth))
    s = {d: alpha * a.get(d, 0) + (1 - alpha) * b.get(d, 0) for d in a.keys() | b.keys()}
    return [{"doc_id": d, "score": v} for d, v in sorted(s.items(), key=lambda x: -x[1])[:k]]

def rrf(query, k=10, c=60, depth=100):
    s = defaultdict(float)
    for hits in (bm25.search(query, depth), semantic.search(query, depth)):
        for rank, h in enumerate(hits, start=1):
            s[h["doc_id"]] += 1 / (c + rank)
    return [{"doc_id": d, "score": v} for d, v in sorted(s.items(), key=lambda x: -x[1])[:k]]
```
</details>

## 4. Move it into `src/discovery/rank/hybrid.py` as `HybridRanker`

The stub is already there with the right signature. Fill in `fit` and `search` (and `params`), then load it here with `discovery.rank.hybrid.HybridRanker`: autoreload picks up your edits.

**Check:** `check_hits` passes below, and in a terminal `IR_SLOW=1 pytest tests/test_methods.py -k hybrid` passes (it was skipped until now).

<details><summary>The search method, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
    def search(self, query, k=10):
        return rrf(query, k) if self.mode == "rrf" else weighted(query, k, self.alpha)
```
</details>

In [ ]:
method = HybridRanker().fit(corpus)
check_hits(method.search(DEV[0], 10), k=10)
method.params()

## 5. Does fusion do what it should?

For each development query, look at where the hybrid top 10 comes from.

**Check:** Repositories both methods found are near the top. If one method fills the whole top 10 at α = 0.5, its scores dominate: check the 0–1 rescaling.

<details><summary>The check, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
for q in DEV:
    A = {h["doc_id"] for h in bm25.search(q)}; B = {h["doc_id"] for h in semantic.search(q)}
    H = [h["doc_id"] for h in rrf(q)]
    both = sum(d in A and d in B for d in H)
    print(f"{q[:40]:40s} | in both: {both} | BM25 only: {sum(d in A and d not in B for d in H)} | semantic only: {sum(d in B and d not in A for d in H)}")
```
</details>

## 6. Read your top 10 on the development queries

Read them. Write down two good results and two bad ones, and why each bad one ranked high (in the markdown cell under the output).

**Check:** you can say in one sentence why each bad result ranked high.

In [ ]:
show_top(method, corpus, DEV)

*Two good results, two bad results, and why:*

- 
- 
- 
- 

## 7. Tune on the 5 development queries only

Grid α for weighted fusion and c for RRF on the development queries. Report both fusions: which wins, and is the winner better than BM25 and semantic alone?

Until Phase 13 gives the development labels, compare settings by reading the top 10 (step 6). When `eval/qrels.csv` has the development labels, use `discovery.evaluation.metrics.ndcg_at_k` and keep the setting with the best mean NDCG@10. **Never tune on the 20 test queries.**

**Check:** a small table: each setting tried, what changed in the top 10 (later: mean NDCG@10). Keep the best and note it in the sheet.

## 8. Speed, results file, hand in

`measure()` times the build and the queries; `save_results()` writes `eval/results/hybrid.json` in the format the Results sheet imports. The metrics stay empty until Phase 13–14: then Member 6's `evaluate_method(method, split="test")` fills them in and you import the file in the **Results sheet** tab of the team plan, add your observations and download the PDF.

**Check:** `eval/results/hybrid.json` exists; this notebook is committed with its outputs on `dev-hybrid`; `pytest` passes.

In [ ]:
speed = measure(lambda: HybridRanker().fit(corpus), DEV)
print(speed)
save_results(method, speed)

## Extra (optional): Fuse three: add TF-IDF

Add TF-IDF as a third list in RRF. Does a third, similar method help, or does it only repeat BM25?

## What we found

*(fill in at the end: build time, the best setting, two good and two bad results, and anything surprising)*